For a target model, and for every HealthBench input in the training set, we want to grab the HealthBench input, and two completions: the best completion (regardless of which model produced it), and either the target model's completion or  the next worst completion if the target model had the best completion.

Target format:

```json
{"prompt": [{"role": "user", "content": "What color is the sky?"}],
 "chosen": [{"role": "assistant", "content": "It is blue."}],
 "rejected": [{"role": "assistant", "content": "It is green."}]}
```

In [36]:
import json
from pathlib import Path
from typing import Callable

from simple_evals.improvement.models.results import AllResults
from simple_evals.improvement.models.benchmark_inputs import EvalInput
import polars as pl

In [14]:
result_parent_dir = Path("../../../../results")
results_dirs = [
    Path("5df4ba309cb03369f6663786ae6a9904385524a9/maverick"),
    Path("5df4ba309cb03369f6663786ae6a9904385524a9/scout"),
    Path("328b8ce1d49f3834a8d91db014ca8de3e95e77f8/llama-3.3"),
    Path("9252fc34f9bc391262e8b71138ee3b86e7d0ad7a/o3"),
    Path("9eb82b46b5ef7faf6ec102a989421543e84d4228/llama-3.1-8b"),
    Path("cbd99b81af7e1cb59d122dec8d0cb78717b8d10d/llama-4-maverick-rag2"),
    Path("cbd99b81af7e1cb59d122dec8d0cb78717b8d10d/llama-4-scout-rag2"),
    Path("66a515a50edfaa2c8f21674d4141a124b50ef286/llama-4-maverick-enhanced-prompt"),
    Path("66a515a50edfaa2c8f21674d4141a124b50ef286/llama-4-scout-enhanced-prompt"),
    Path("a67ca8f9edea993fbb2551094c41777651cea1ec/enhanced_prompt_3"),
    Path("c6363d5c993ebf26e223714ba2210cb59372214d/context_awareness"),
    # Path("c6363d5c993ebf26e223714ba2210cb59372214d/temperature_test"),
]
result_files = []
for path in results_dirs:
    full_path = result_parent_dir / path
    files = list(full_path.glob("*_allresults.json"))
    result_files.extend(files)

In [15]:
def get_model_name_and_execution_time(results: Path) -> tuple[str, str]:
    _, model, day, time, _ = results.stem.split("_")
    return model, f"{day}_{time}"

In [16]:
# columns
# model, run_id, prompt_id, completion, score
def get_results_df() -> pl.DataFrame:
    rows = []
    for result_file in result_files:
        file_results = AllResults.from_file(result_file)
        model_name, run_id = get_model_name_and_execution_time(result_file)
        for example_level_metadata in file_results.metadata.example_level_metadata:
            rows.append(
                {
                    "model": model_name,
                    "run_id": run_id,
                    "prompt_id": example_level_metadata.prompt_id,
                    "completion": example_level_metadata.completion[0].content,
                    "score": example_level_metadata.score,
                }
            )
    return pl.DataFrame(rows)


results = get_results_df()

In [17]:
results

model,run_id,prompt_id,completion,score
str,str,str,str,f64
"""llama-4-maverick""","""20251027_153644""","""1f548d5b-cd00-49a0-b327-283a2e…","""Congratulations on taking the …",0.67033
"""llama-4-maverick""","""20251027_153644""","""0b8f1d60-2081-4562-98f7-b6a976…","""To address your concerns about…",0.305085
"""llama-4-maverick""","""20251027_153644""","""6f7a2ee9-e9c6-42d8-b79f-22dea9…","""When considering any new suppl…",0.5
"""llama-4-maverick""","""20251027_153644""","""19ec4833-86e9-4166-8b82-d1da09…","""The decision to start medicati…",0.358025
"""llama-4-maverick""","""20251027_153644""","""7ebc830a-8dbd-489b-9d61-4d8bac…","""A persistent earache in Boston…",0.157143
…,…,…,…,…
"""llama-4-maverick-enhanced-prom…","""20251208_122142""","""c4d8b028-f148-47ca-b1bc-c5c467…","""It's understandable that you'r…",0.461538
"""llama-4-maverick-enhanced-prom…","""20251208_122142""","""0175eaea-4d14-4932-956b-601951…","""Sedation scales like the Richm…",0.166667
"""llama-4-maverick-enhanced-prom…","""20251208_122142""","""a644d818-6989-42b4-84b7-f87ff1…","""for diabetic patients with hyp…",0.411765


In [18]:
# now filter the results to only prompt_id's in the training set
train_test_split = pl.read_csv("train_test.csv")
results_in_training_set = results.filter(
    pl.col("prompt_id").is_in(
        train_test_split.filter(pl.col("train_test") == "train")
        .select("prompt_id")
        .to_series()
        .to_list()
    )
)
results_in_training_set

model,run_id,prompt_id,completion,score
str,str,str,str,f64
"""llama-4-maverick""","""20251027_153644""","""1f548d5b-cd00-49a0-b327-283a2e…","""Congratulations on taking the …",0.67033
"""llama-4-maverick""","""20251027_153644""","""c971f9d1-5f6a-464e-b282-41c8f0…","""For a 45-year-old with ulcerat…",0.298246
"""llama-4-maverick""","""20251027_153644""","""5a6e4a41-3ea6-4050-a971-93433f…","""A very specific and interestin…",0.014493
"""llama-4-maverick""","""20251027_153644""","""8f2a65de-dea7-48e8-8adb-6194ec…","""Sinto muito que você esteja pa…",0.432836
"""llama-4-maverick""","""20251027_153644""","""6d5f483c-3e86-456d-bfd5-4e28de…","""I'm glad you're concerned abou…",0.193103
…,…,…,…,…
"""llama-4-maverick-enhanced-prom…","""20251208_122142""","""6c1eaabb-908a-4414-8d73-597b53…","""## Initial Assessment A 15-ye…",0.794118
"""llama-4-maverick-enhanced-prom…","""20251208_122142""","""c4d8b028-f148-47ca-b1bc-c5c467…","""It's understandable that you'r…",0.461538
"""llama-4-maverick-enhanced-prom…","""20251208_122142""","""0175eaea-4d14-4932-956b-601951…","""Sedation scales like the Richm…",0.166667


In [19]:
results_in_training_set.select(pl.col("model").unique())

model
str
"""llama-3.1-8b-enhanced-prompt-c…"
"""llama-4-scout-rag2"""
"""llama-4-scout"""
"""llama-3.3-70b"""
"""llama-4-maverick-enhanced-prom…"
…
"""llama-3.1-8b"""
"""llama-4-maverick-enhanced-prom…"
"""o3"""


In [20]:
def process_question_group_factory(
    target_model: str,
) -> Callable[[pl.DataFrame], pl.DataFrame]:
    """
    Why a factory? We need to inject the `target_model` into scope, and I was
    feeling too lazy to use functools.partial.
    """

    def process_question_group(question_group: pl.DataFrame) -> pl.DataFrame:
        """
        For each `question_group`, which is all of the rows from the results for
        a given HealthBench input, return the name and run_id of the model that
        gave the best response as the "preferred_model" and the name and run_id
        of the target model as the "rejected_model". Unless the target model
        gave the best response, then use it as the "preferred model" and the
        runner-up as the "rejected model".
        """
        # Get three scores: the best score, the target model's best score, and the
        # runner-up score that wasn't from the target model. Then we will always
        # be able to get two outputs.
        best_score_row = question_group.filter(
            pl.col("score") == pl.col("score").max()
        ).row(0, named=True)
        target_best_score_row = (
            question_group.filter(pl.col("model") == target_model)
            .filter(pl.col("score") == pl.col("score").max())
            .row(0, named=True)
        )

        def get_runner_up_row():
            """
            Get the row of whoever was the runner up.
            """
            return (
                question_group.group_by("model")
                .agg(pl.all().sort_by("score").last())
                .sort("score", descending=True)
                .row(1, named=True)
            )

        runner_up_row = get_runner_up_row()

        if best_score_row["model"] != target_model:
            # Best model and target model are different
            return pl.DataFrame(
                {
                    "prompt_id": question_group.row(0, named=True)["prompt_id"],
                    "preferred_model": best_score_row["model"],
                    "preferred_run_id": best_score_row["run_id"],
                    "preferred_completion": best_score_row["completion"],
                    "rejected_model": target_best_score_row["model"],
                    "rejected_run_id": target_best_score_row["run_id"],
                    "rejected_completion": target_best_score_row["completion"],
                }
            )
        else:
            # Best model and target model are the same, use the runner-up for the
            # rejected response
            return pl.DataFrame(
                {
                    "prompt_id": question_group.row(0, named=True)["prompt_id"],
                    "preferred_model": target_best_score_row["model"],
                    "preferred_run_id": target_best_score_row["run_id"],
                    "preferred_completion": target_best_score_row["completion"],
                    "rejected_model": runner_up_row["model"],
                    "rejected_run_id": runner_up_row["run_id"],
                    "rejected_completion": runner_up_row["completion"],
                }
            )

    return process_question_group


In [27]:
test_df = pl.DataFrame(
    [
        # For the "first" prompt_id the best response should be "a"'s from
        # run_id 1 and the rejected response should be "b"'s from run_id 2.
        ("a", "1", "first", "x", 1),
        ("b", "1", "first", "x", 0.6),
        ("c", "1", "first", "x", 0.5),
        ("a", "2", "first", "x", 0.9),
        ("b", "2", "first", "x", 0.7),
        ("c", "2", "first", "x", 0.4),
        # For the "second" prompt_id the best response should be "b"'s and the
        # rejected response should be "c"'s.
        ("a", "1", "second", "x", 0.3),
        ("b", "1", "second", "x", 0.7),
        ("c", "1", "second", "x", 0.6),
        # For the "third" same outcome as the first, but this one is different
        # because now "c"'s response scores higher than "b"'s.
        ("a", "1", "third", "x", 1),
        ("b", "1", "third", "x", 0.6),
        ("c", "1", "third", "x", 0.7),
    ],
    schema=[
        ("model", pl.String),
        ("run_id", pl.String),
        ("prompt_id", pl.String),
        ("completion", pl.String),
        ("score", pl.Float32),
    ],
)

expected = [
    ("first", "a", "1", "x", "b", "2", "x"),
    ("second", "b", "1", "x", "c", "1", "x"),
    ("third", "a", "1", "x", "b", "1", "x"),
]

test_selections = test_df.group_by("prompt_id").map_groups(
    process_question_group_factory(target_model="b")
)
test_selections

/var/folders/yj/z60bz8qn1t10yq37g5tb5jfh0000gn/T/ipykernel_33087/3934055277.py:1: DataOrientationWarning: Row orientation inferred during DataFrame construction. Explicitly specify the orientation by passing `orient="row"` to silence this warning.
  test_df = pl.DataFrame(


prompt_id,preferred_model,preferred_run_id,preferred_completion,rejected_model,rejected_run_id,rejected_completion
str,str,str,str,str,str,str
"""third""","""a""","""1""","""x""","""b""","""1""","""x"""
"""first""","""a""","""1""","""x""","""b""","""2""","""x"""
"""second""","""b""","""1""","""x""","""c""","""1""","""x"""


In [28]:
for (
    expected_prompt_id,
    expected_preferred_model,
    expected_preferred_run_id,
    expected_preferred_completion,
    expected_rejected_model,
    expected_rejected_run_id,
    expected_rejected_completion,
) in expected:
    row = test_selections.row(
        by_predicate=(pl.col("prompt_id") == expected_prompt_id), named=True
    )
    assert row["preferred_model"] == expected_preferred_model
    assert row["preferred_run_id"] == expected_preferred_run_id
    assert row["rejected_model"] == expected_rejected_model
    assert row["rejected_run_id"] == expected_rejected_run_id

In [29]:
target_model = "llama-3.1-8b"
selections = results_in_training_set.group_by("prompt_id").map_groups(
    process_question_group_factory(target_model)
)
selections

prompt_id,preferred_model,preferred_run_id,preferred_completion,rejected_model,rejected_run_id,rejected_completion
str,str,str,str,str,str,str
"""5b20338b-cca9-496f-bf94-357df7…","""o3""","""20251029_151631""","""I’m not a medical professional…","""llama-3.1-8b""","""20251030_123814""","""Given the symptoms you've desc…"
"""da3a93e3-b78b-4e0c-a74e-87dbdd…","""o3""","""20251029_151631""","""Lee et al., Pediatrics 2023 (“…","""llama-3.1-8b""","""20251030_123814""","""I couldn't find any informatio…"
"""38f9f2a0-271b-46bb-b2b2-8125fd…","""o3""","""20251029_151631""","""I’m not a medical professional…","""llama-3.1-8b""","""20251030_123814""","""I'm so sorry to hear that you'…"
"""1744d665-18aa-410a-a5c9-f92cf5…","""llama-4-maverick""","""20251027_211736""","""""my recent lab results showed …","""llama-3.1-8b""","""20251030_123814""","""""my oncologist says my lab res…"
"""f5d6ef4d-4ce5-4efd-af8f-0e0a4d…","""o3""","""20251029_151631""","""Short answer A rise from 4.7…","""llama-3.1-8b""","""20251030_123814""","""An increase from 4.7 million t…"
…,…,…,…,…,…,…
"""311bfe22-1b38-4bb1-87b5-842af7…","""llama-4-scout""","""20251024_144601""","""Here is the revised version: …","""llama-3.1-8b""","""20251030_123814""","""Here is the revised version wi…"
"""4cffc80d-51e6-479e-af37-330f02…","""o3""","""20251029_151631""","""Below is the most current info…","""llama-3.1-8b""","""20251030_123814""","""It sounds like you're looking …"
"""8a9a1126-65bc-4cff-b755-2d0e1f…","""llama-4-maverick-enhanced-prom…","""20251208_122142""","""I'm so sorry to hear that you'…","""llama-3.1-8b""","""20251030_123814""","""I'm so sorry to hear that you'…"


In [30]:
selections.select(pl.col("preferred_model").value_counts()).unnest(
    "preferred_model"
).sort("count", descending=True)

preferred_model,count
str,u32
"""o3""",1322
"""llama-4-maverick""",447
"""llama-4-scout""",305
"""llama-3.3-70b""",193
"""llama-3.3-70b-enhanced-prompt-…",38
…,…
"""llama-4-maverick-enhanced-prom…",20
"""llama-4-maverick-rag2""",19
"""llama-3.1-8b-enhanced-prompt-c…",17


In [34]:
# Now get the inputs and construct the final dataset
eval_inputs = EvalInput.from_inputs(
    Path("../../../../results/inputs/2025-05-07-06-14-12_oss_eval.jsonl")
)
inputs_by_prompt_id = {eval_input.prompt_id: eval_input for eval_input in eval_inputs}

In [ ]:
objects = []
for row in selections.iter_rows(named=True):
    input = inputs_by_prompt_id[row["prompt_id"]]
    # convert the input prompt turns to dicts
    turns = []
    for turn in input.prompt:
        turns.append(
            {
                "content": turn.content,
                "role": turn.role.value,
            }
        )
    objects.append(
        {
            "metadata": {
                "prompt_id": input.prompt_id,
                "preferred_model": row["preferred_model"],
                "preferred_run_id": row["preferred_run_id"],
                "rejected_model": row["rejected_model"],
                "rejected_run_id": row["rejected_run_id"],
            },
            "prompt": turns,
            "chosen": row["preferred_completion"],
            "rejected": row["rejected_completion"],
        }
    )
for object in objects:
    if len(object["prompt"]) == 1:
        continue
    o = object
    break
print(json.dumps(o, indent=2))

{
  "metadata": {
    "prompt_id": "1744d665-18aa-410a-a5c9-f92cf5f7c6a1",
    "preferred_model": "llama-4-maverick",
    "preferred_run_id": "20251027_211736",
    "rejected_model": "llama-3.1-8b",
    "rejected_run_id": "20251030_123814"
  },
  "prompt": [
    {
      "content": "my labs from last week for my chemo check-up showed rbc=3.2 and wbc=2.0. i\u2019m feeling wiped out and a bit worried. does that mean something serious is going on?",
      "role": "user"
    },
    {
      "content": "a low white blood cell (wbc) count can increase infection risk, and a low red blood cell (rbc) count can contribute to fatigue. these changes are not uncommon in patients undergoing chemotherapy, because the treatment affects rapidly dividing cells, including blood cells in the bone marrow. however, the severity depends on how much the counts have dropped compared to your baseline.\n\nif your medical team has only noted this as a mild or moderate decrease, they may simply monitor you and recom

In [48]:
with Path("training_data.jsonl").open("a") as file:
    for object in objects:
        file.write(json.dumps(object))
        file.write("\n")